# Offline Reinforcement Learning on a 10 × 10 GridWorld

## Portfolio Project

This notebook demonstrates offline RL on a simple problem where every part of the learning process can be inspected directly.

```text
GridWorld
   ↓
Fixed CSV trajectories
   ↓
Dataset inspection
   ↓
Offline Q-learning from scratch
   ↓
Gymnasium version of the same problem
   ↓
Evaluation and comparison
```

The first implementation intentionally contains no imports.

## 1. What Is the Environment?

The agent moves in a 10 × 10 grid.

```text
Start = (0,0)
Goal  = (9,9)
```

Actions:

```text
0 Up
1 Right
2 Down
3 Left
```

## 2. Grid Layout

```text
    0 1 2 3 4 5 6 7 8 9
0   S . . . . . . . . .
1   . . # . . . . . . .
2   . . # . . . # . # .
3   . . # . . . # . # .
4   . . # # # # # . # .
5   . . . . . . # . # .
6   . . . . . . # . . .
7   . . . . . . # # # .
8   . . . . . . . . . .
9   . . . . . . . . . G
```

`#` means blocked.

## 3. What Is the Dataset About?

The CSV stores trajectories that were collected before training.

Each row is one transition:

```text
state
→ action
→ reward
→ next state
→ done information
```

The learner is not allowed to collect new transitions during training.

## 4. What Is the Agent Trying to Learn?

The policy should answer:

> Given the current grid position, which direction should I move so that I reach `(9,9)` while avoiding blocked cells and boundaries?

The policy is a mapping:

```text
state → action
```

## 5. State Space

The state is `(row, column)`.

For tabular learning:

```text
State_ID = row * 10 + column
```

Examples:

```text
(0,0) → 0
(1,0) → 10
(9,9) → 99
```

There are 100 possible encoded states.

## 6. Action Space

| ID | Action |
|---:|---|
| 0 | Up |
| 1 | Right |
| 2 | Down |
| 3 | Left |

There are four discrete actions.

## 7. Reward Function

```text
Normal valid move       0
Blocked cell hit       -5
Boundary collision     -5
Goal reached           +10
```

A collision leaves the agent in the same state.

## 8. Episode Ending

```text
Terminated → goal reached
Truncated  → 100-step limit reached
Done       → Terminated OR Truncated
```

## 9. How Was the Offline Dataset Generated?

The behavior policy deliberately contains both useful and poor actions.

```text
793 mixed-guided episodes
207 pure-random episodes
```

Mixed-guided episodes use:

```text
65% distance-reducing action
35% random action
```

Actual fixed dataset:

```text
1,000 episodes
42,986 transitions
797 successful episodes
203 truncated episodes
```

# Part I — Inspect the Dataset with Plain Python

No import statements are used in this section.

## 10. Dataset Path

For the repository:

```text
../data/gridworld_offline_dataset.csv
```

For Colab, upload the CSV and use:

```text
/content/gridworld_offline_dataset.csv
```

In [54]:
# No imports.

#DATASET_FILE = "../data/gridworld_offline_dataset.csv"

In [28]:
# Upload the offline RL dataset from your computer.

from google.colab import files

uploaded = files.upload()

# Automatically use the uploaded file name.
DATASET_FILE = next(iter(uploaded))

print("Dataset file:")
print(DATASET_FILE)

Saving gridworld_offline_dataset.csv to gridworld_offline_dataset (1).csv
Dataset file:
gridworld_offline_dataset (1).csv


## 11. Print the CSV Header and First Row

In [29]:
file = open(DATASET_FILE, "r")

header_line = file.readline().strip()
first_line = file.readline().strip()

file.close()

header = header_line.split(",")
first_row = first_line.split(",")

print("Columns:", len(header))

for index in range(len(header)):
    print(index, "->", header[index])

print("\nFirst row:")
print(first_row)

Columns: 20
0 -> Episode
1 -> Step
2 -> Episode_Behavior_Type
3 -> State_Row
4 -> State_Col
5 -> State_ID
6 -> Action_ID
7 -> Action_Name
8 -> Action_Source
9 -> Reward
10 -> Next_State_Row
11 -> Next_State_Col
12 -> Next_State_ID
13 -> Terminated
14 -> Truncated
15 -> Done
16 -> Hit_Blocked
17 -> Hit_Boundary
18 -> Distance_To_Goal
19 -> Next_Distance_To_Goal

First row:
['1', '0', 'Mixed Guided', '0', '0', '0', '2', 'Down', 'Guided', '0', '1', '0', '10', '0', '0', '0', '0', '0', '18', '17']


## 12. Locate the Core Offline RL Fields

The minimum learning tuple is:

```text
State_ID
Action_ID
Reward
Next_State_ID
Done
```

In [30]:
STATE_ID_INDEX = header.index("State_ID")
ACTION_ID_INDEX = header.index("Action_ID")
REWARD_INDEX = header.index("Reward")
NEXT_STATE_ID_INDEX = header.index("Next_State_ID")
TERMINATED_INDEX = header.index("Terminated")
TRUNCATED_INDEX = header.index("Truncated")
DONE_INDEX = header.index("Done")

print("State:", STATE_ID_INDEX)
print("Action:", ACTION_ID_INDEX)
print("Reward:", REWARD_INDEX)
print("Next state:", NEXT_STATE_ID_INDEX)
print("Terminated:", TERMINATED_INDEX)
print("Truncated:", TRUNCATED_INDEX)
print("Done:", DONE_INDEX)

State: 5
Action: 6
Reward: 9
Next state: 12
Terminated: 13
Truncated: 14
Done: 15


## 13. Scan the Full Dataset

We inspect:

- number of transitions
- number of episodes
- action counts
- reward counts
- terminal/truncation counts
- collision counts
- state coverage
- state-action coverage

In [31]:
transition_count = 0

episodes = set()
states = set()
state_actions = set()

action_counts = {
    0: 0,
    1: 0,
    2: 0,
    3: 0,
}

reward_counts = {}

terminated_count = 0
truncated_count = 0
done_count = 0
blocked_hits = 0
boundary_hits = 0

file = open(DATASET_FILE, "r")
header = file.readline().strip().split(",")

for line in file:

    parts = line.strip().split(",")

    transition_count += 1

    episode_id = int(parts[header.index("Episode")])
    state_id = int(parts[header.index("State_ID")])
    action_id = int(parts[header.index("Action_ID")])
    reward = int(parts[header.index("Reward")])

    episodes.add(episode_id)
    states.add(state_id)
    state_actions.add((state_id, action_id))

    action_counts[action_id] += 1

    reward_counts[reward] = (
        reward_counts.get(reward, 0)
        +
        1
    )

    terminated_count += int(parts[header.index("Terminated")])
    truncated_count += int(parts[header.index("Truncated")])
    done_count += int(parts[header.index("Done")])

    blocked_hits += int(parts[header.index("Hit_Blocked")])
    boundary_hits += int(parts[header.index("Hit_Boundary")])

file.close()

print("Transitions:", transition_count)
print("Episodes:", len(episodes))
print("Observed states:", len(states))
print("Observed state-action pairs:", len(state_actions))
print("Maximum state-action pairs:", 100 * 4)
print("\nAction counts:", action_counts)
print("Reward counts:", reward_counts)
print("\nTerminated:", terminated_count)
print("Truncated:", truncated_count)
print("Done:", done_count)
print("\nBlocked hits:", blocked_hits)
print("Boundary hits:", boundary_hits)

Transitions: 42986
Episodes: 1000
Observed states: 80
Observed state-action pairs: 320
Maximum state-action pairs: 400

Action counts: {0: 7214, 1: 14240, 2: 14273, 3: 7259}
Reward counts: {0: 34874, -5: 7315, 10: 797}

Terminated: 797
Truncated: 203
Done: 1000

Blocked hits: 2842
Boundary hits: 4473


## 14. Episode-Level Statistics

In [32]:
episode_returns = {}
episode_lengths = {}

file = open(DATASET_FILE, "r")
header = file.readline().strip().split(",")

for line in file:

    parts = line.strip().split(",")

    episode_id = int(parts[header.index("Episode")])
    reward = int(parts[header.index("Reward")])

    episode_returns[episode_id] = (
        episode_returns.get(episode_id, 0)
        +
        reward
    )

    episode_lengths[episode_id] = (
        episode_lengths.get(episode_id, 0)
        +
        1
    )

file.close()

returns = list(episode_returns.values())
lengths = list(episode_lengths.values())

print("Episodes:", len(returns))
print("Min return:", min(returns))
print("Max return:", max(returns))
print("Average return:", sum(returns) / len(returns))
print("Min length:", min(lengths))
print("Max length:", max(lengths))
print("Average length:", sum(lengths) / len(lengths))

Episodes: 1000
Min return: -230
Max return: 10
Average return: -28.605
Min length: 18
Max length: 100
Average length: 42.986


## 15. Print One Complete Transition

In [33]:
file = open(DATASET_FILE, "r")
header = file.readline().strip().split(",")
parts = file.readline().strip().split(",")
file.close()

state = (
    int(parts[header.index("State_Row")]),
    int(parts[header.index("State_Col")]),
)

action_id = int(parts[header.index("Action_ID")])
action_name = parts[header.index("Action_Name")]
reward = int(parts[header.index("Reward")])

next_state = (
    int(parts[header.index("Next_State_Row")]),
    int(parts[header.index("Next_State_Col")]),
)

terminated = bool(int(parts[header.index("Terminated")]))
truncated = bool(int(parts[header.index("Truncated")]))
done = bool(int(parts[header.index("Done")]))

print("State:", state)
print("Action:", action_id, action_name)
print("Reward:", reward)
print("Next state:", next_state)
print("Terminated:", terminated)
print("Truncated:", truncated)
print("Done:", done)

State: (0, 0)
Action: 2 Down
Reward: 0
Next state: (1, 0)
Terminated: False
Truncated: False
Done: False


# Part II — Offline Q-Learning from Scratch

## 16. Method Choice

For this small discrete problem, a Q-table is the clearest method.

To make it explicitly offline, the Bellman backup is restricted to actions that were observed in the fixed dataset.

## 17. Pros and Cons

### Pros

- no RL module
- fully transparent
- uses fixed data only
- simple Bellman updates
- appropriate for a 100-state environment

### Cons

- no generalization
- dataset coverage matters
- not suitable for large continuous problems

## 18. Create the Q-Table

In [34]:
NUM_STATES = 100
NUM_ACTIONS = 4

Q = []

for state_id in range(NUM_STATES):

    Q.append([
        0.0,
        0.0,
        0.0,
        0.0,
    ])

print("Q shape:", len(Q), "x", len(Q[0]))

Q shape: 100 x 4


## 19. Find the Actions Observed at Each State

In [35]:
observed_actions = {}

file = open(DATASET_FILE, "r")
header = file.readline().strip().split(",")

for line in file:

    parts = line.strip().split(",")

    state_id = int(parts[header.index("State_ID")])
    action_id = int(parts[header.index("Action_ID")])

    if state_id not in observed_actions:
        observed_actions[state_id] = set()

    observed_actions[state_id].add(action_id)

file.close()

print("States with support:", len(observed_actions))
print("Actions observed at start:", observed_actions.get(0))

States with support: 80
Actions observed at start: {0, 1, 2, 3}


## 20. Training Hyperparameters

In [36]:
LEARNING_RATE = 0.10
GAMMA = 0.95
EPOCHS = 15

## 21. Train Only from the Fixed CSV

There is no environment interaction in this training loop.

In [37]:
training_history = []

for epoch in range(EPOCHS):

    absolute_td_error = 0.0
    number_of_updates = 0

    file = open(DATASET_FILE, "r")
    header = file.readline().strip().split(",")

    for line in file:

        parts = line.strip().split(",")

        state_id = int(parts[header.index("State_ID")])
        action_id = int(parts[header.index("Action_ID")])
        reward = float(parts[header.index("Reward")])
        next_state_id = int(parts[header.index("Next_State_ID")])
        done = bool(int(parts[header.index("Done")]))

        if done:
            next_value = 0.0

        else:
            supported = observed_actions.get(
                next_state_id,
                set()
            )

            if supported:
                next_value = max(
                    Q[next_state_id][next_action]
                    for next_action in supported
                )
            else:
                next_value = 0.0

        target = reward + GAMMA * next_value

        td_error = (
            target
            -
            Q[state_id][action_id]
        )

        Q[state_id][action_id] += (
            LEARNING_RATE
            *
            td_error
        )

        absolute_td_error += abs(td_error)
        number_of_updates += 1

    file.close()

    mean_td_error = (
        absolute_td_error
        /
        number_of_updates
    )

    training_history.append(mean_td_error)

    print(
        "Epoch",
        epoch + 1,
        "Mean absolute TD error:",
        round(mean_td_error, 6)
    )

Epoch 1 Mean absolute TD error: 0.473157
Epoch 2 Mean absolute TD error: 0.102486
Epoch 3 Mean absolute TD error: 0.063091
Epoch 4 Mean absolute TD error: 0.062142
Epoch 5 Mean absolute TD error: 0.062046
Epoch 6 Mean absolute TD error: 0.06203
Epoch 7 Mean absolute TD error: 0.062027
Epoch 8 Mean absolute TD error: 0.062026
Epoch 9 Mean absolute TD error: 0.062026
Epoch 10 Mean absolute TD error: 0.062026
Epoch 11 Mean absolute TD error: 0.062026
Epoch 12 Mean absolute TD error: 0.062026
Epoch 13 Mean absolute TD error: 0.062026
Epoch 14 Mean absolute TD error: 0.062026
Epoch 15 Mean absolute TD error: 0.062026


## 22. Inspect Q-Values at the Start State

In [38]:
ACTION_NAMES = [
    "Up",
    "Right",
    "Down",
    "Left",
]

for action_id in range(4):

    print(
        action_id,
        ACTION_NAMES[action_id],
        "->",
        round(Q[0][action_id], 4)
    )

0 Up -> -1.0417
1 Right -> 4.1585
2 Down -> 4.1642
3 Left -> -1.0534


## 23. Extract the Greedy Supported Policy

In [39]:
policy = {}

for state_id in observed_actions:

    supported = observed_actions[state_id]

    best_action = max(
        supported,
        key=lambda action_id: Q[state_id][action_id]
    )

    policy[state_id] = best_action

print("Policy states:", len(policy))

Policy states: 80


## 24. Print the Policy on the Grid

```text
^ Up
> Right
v Down
< Left
# Blocked
```

In [40]:
BLOCKED_CELLS = {
    (1, 2), (2, 2), (3, 2), (4, 2),
    (4, 3), (4, 4), (4, 5), (4, 6),
    (2, 6), (3, 6), (5, 6), (6, 6),
    (7, 6), (7, 7), (7, 8),
    (2, 8), (3, 8), (4, 8), (5, 8),
}

SYMBOLS = {
    0: "^",
    1: ">",
    2: "v",
    3: "<",
}

for row in range(10):

    output = []

    for col in range(10):

        position = (row, col)
        state_id = row * 10 + col

        if position == (0, 0):
            symbol = "S"

        elif position == (9, 9):
            symbol = "G"

        elif position in BLOCKED_CELLS:
            symbol = "#"

        elif state_id in policy:
            symbol = SYMBOLS[policy[state_id]]

        else:
            symbol = "?"

        output.append(symbol)

    print(" ".join(output))

S v > > v > v > > v
v v # > > > > v > v
> v # > ^ ^ # v # v
> v # ^ ^ ^ # v # v
> v # # # # # v # v
> v v > v v # v # v
v > v v v v # > > v
> > v > > v # # # v
> > > > > > > > v v
> > > > > > > > > G


## 25. Define Plain-Python Dynamics for Evaluation

Training has already finished.

This function is only used to test the learned policy.

In [41]:
ACTION_CHANGES = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}

def plain_step(state, action_id):

    dr, dc = ACTION_CHANGES[action_id]

    candidate = (
        state[0] + dr,
        state[1] + dc,
    )

    inside = (
        0 <= candidate[0] < 10
        and
        0 <= candidate[1] < 10
    )

    if (
        not inside
        or
        candidate in BLOCKED_CELLS
    ):
        return state, -5, False

    if candidate == (9, 9):
        return candidate, 10, True

    return candidate, 0, False

## 26. Evaluate the From-Scratch Policy

In [42]:
state = (0, 0)
path = [state]
total_reward = 0
reached_goal = False

for step_number in range(100):

    state_id = state[0] * 10 + state[1]

    if state_id not in policy:
        break

    action_id = policy[state_id]

    next_state, reward, terminated = plain_step(
        state,
        action_id,
    )

    total_reward += reward
    path.append(next_state)
    state = next_state

    if terminated:
        reached_goal = True
        break

print("Reached goal:", reached_goal)
print("Total reward:", total_reward)
print("Steps:", len(path) - 1)

print("\nPath:")
for position in path:
    print(position)

Reached goal: True
Total reward: 10
Steps: 18

Path:
(0, 0)
(1, 0)
(2, 0)
(2, 1)
(3, 1)
(4, 1)
(5, 1)
(6, 1)
(6, 2)
(7, 2)
(8, 2)
(8, 3)
(8, 4)
(8, 5)
(8, 6)
(8, 7)
(8, 8)
(9, 8)
(9, 9)


# Part III — Same Offline RL Problem with Gymnasium

## 27. Role of Gymnasium

Gymnasium gives the problem a standard environment interface.

Training still comes from the same fixed CSV.

```text
Gymnasium = environment definition and evaluation
CSV       = offline training experience
```

## 28. Pros and Cons of the Gymnasium Version

### Pros

- standard environment API
- explicit observation/action spaces
- reusable with other RL libraries
- easier evaluation

### Cons

- extra dependency
- Gymnasium is not itself an offline RL algorithm

## 29. Install Packages

In [43]:
# Uncomment if required.

# %pip install gymnasium numpy

## 30. Import Gymnasium and NumPy

In [44]:
import csv

import gymnasium as gym

from gymnasium import spaces

import numpy as np

print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)

Gymnasium: 1.3.0
NumPy: 2.1.3


## 31. Define the Same Grid as a Gymnasium Environment

In [45]:
class OfflineGridWorld10x10(gym.Env):

    metadata = {
        "render_modes": [
            "ansi"
        ]
    }

    def __init__(self):

        super().__init__()

        self.start = (0, 0)
        self.goal = (9, 9)

        self.blocked = {
            (1, 2), (2, 2), (3, 2), (4, 2),
            (4, 3), (4, 4), (4, 5), (4, 6),
            (2, 6), (3, 6), (5, 6), (6, 6),
            (7, 6), (7, 7), (7, 8),
            (2, 8), (3, 8), (4, 8), (5, 8),
        }

        self.changes = {
            0: (-1, 0),
            1: (0, 1),
            2: (1, 0),
            3: (0, -1),
        }

        self.observation_space = spaces.Discrete(100)
        self.action_space = spaces.Discrete(4)

        self.position = self.start

    def _state_id(self, position):

        return (
            position[0] * 10
            +
            position[1]
        )

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.position = self.start

        return (
            self._state_id(self.position),
            {
                "position": self.position
            },
        )

    def step(self, action):

        dr, dc = self.changes[int(action)]

        candidate = (
            self.position[0] + dr,
            self.position[1] + dc,
        )

        inside = (
            0 <= candidate[0] < 10
            and
            0 <= candidate[1] < 10
        )

        hit_blocked = candidate in self.blocked
        hit_boundary = not inside

        if hit_blocked or hit_boundary:

            next_position = self.position
            reward = -5
            terminated = False

        elif candidate == self.goal:

            next_position = candidate
            reward = 10
            terminated = True

        else:

            next_position = candidate
            reward = 0
            terminated = False

        self.position = next_position

        observation = self._state_id(
            self.position
        )

        truncated = False

        info = {
            "position": self.position,
            "hit_blocked": hit_blocked,
            "hit_boundary": hit_boundary,
        }

        return (
            observation,
            reward,
            terminated,
            truncated,
            info,
        )

    def render(self):

        rows = []

        for row in range(10):

            symbols = []

            for col in range(10):

                position = (row, col)

                if position == self.position:
                    symbol = "A"

                elif position == self.goal:
                    symbol = "G"

                elif position in self.blocked:
                    symbol = "#"

                else:
                    symbol = "."

                symbols.append(symbol)

            rows.append(
                " ".join(symbols)
            )

        return "\n".join(rows)

## 32. Inspect the Gymnasium Spaces

In [46]:
env = OfflineGridWorld10x10()

print("Observation space:")
print(env.observation_space)

print("\nAction space:")
print(env.action_space)

observation, info = env.reset()

print("\nInitial observation:")
print(observation)

print("\nInitial position:")
print(info["position"])

print("\nGrid:")
print(env.render())

Observation space:
Discrete(100)

Action space:
Discrete(4)

Initial observation:
0

Initial position:
(0, 0)

Grid:
A . . . . . . . . .
. . # . . . . . . .
. . # . . . # . # .
. . # . . . # . # .
. . # # # # # . # .
. . . . . . # . # .
. . . . . . # . . .
. . . . . . # # # .
. . . . . . . . . .
. . . . . . . . . G


## 33. Load the Same Fixed CSV

No Gymnasium environment calls are used to create training transitions here.

In [47]:
offline_states = []
offline_actions = []
offline_rewards = []
offline_next_states = []
offline_dones = []

with open(
    DATASET_FILE,
    "r",
    newline="",
) as file:

    reader = csv.DictReader(file)

    for row in reader:

        offline_states.append(
            int(row["State_ID"])
        )

        offline_actions.append(
            int(row["Action_ID"])
        )

        offline_rewards.append(
            float(row["Reward"])
        )

        offline_next_states.append(
            int(row["Next_State_ID"])
        )

        offline_dones.append(
            bool(int(row["Done"]))
        )

offline_states = np.asarray(
    offline_states,
    dtype=np.int64,
)

offline_actions = np.asarray(
    offline_actions,
    dtype=np.int64,
)

offline_rewards = np.asarray(
    offline_rewards,
    dtype=np.float64,
)

offline_next_states = np.asarray(
    offline_next_states,
    dtype=np.int64,
)

offline_dones = np.asarray(
    offline_dones,
    dtype=bool,
)

print("States:", offline_states.shape)
print("Actions:", offline_actions.shape)
print("Rewards:", offline_rewards.shape)
print("Next states:", offline_next_states.shape)
print("Done:", offline_dones.shape)

States: (42986,)
Actions: (42986,)
Rewards: (42986,)
Next states: (42986,)
Done: (42986,)


## 34. Build Offline Action Support

In [48]:
gym_observed_actions = {}

for index in range(len(offline_states)):

    state_id = int(
        offline_states[index]
    )

    action_id = int(
        offline_actions[index]
    )

    if state_id not in gym_observed_actions:
        gym_observed_actions[state_id] = set()

    gym_observed_actions[state_id].add(
        action_id
    )

print(
    "States with support:",
    len(gym_observed_actions)
)

States with support: 80


## 35. Train the Same Offline Q-Learning Rule with NumPy

The environment is still not used during training.

In [49]:
gym_q = np.zeros(
    (
        env.observation_space.n,
        env.action_space.n,
    ),
    dtype=np.float64,
)

GYM_ALPHA = 0.10
GYM_GAMMA = 0.95
GYM_EPOCHS = 15

for epoch in range(GYM_EPOCHS):

    absolute_td_error = 0.0

    for index in range(len(offline_states)):

        state_id = int(
            offline_states[index]
        )

        action_id = int(
            offline_actions[index]
        )

        reward = float(
            offline_rewards[index]
        )

        next_state_id = int(
            offline_next_states[index]
        )

        done = bool(
            offline_dones[index]
        )

        if done:

            next_value = 0.0

        else:

            supported = gym_observed_actions.get(
                next_state_id,
                set(),
            )

            if supported:

                next_value = max(
                    gym_q[
                        next_state_id,
                        next_action,
                    ]
                    for next_action in supported
                )

            else:

                next_value = 0.0

        target = (
            reward
            +
            GYM_GAMMA
            *
            next_value
        )

        td_error = (
            target
            -
            gym_q[
                state_id,
                action_id,
            ]
        )

        gym_q[
            state_id,
            action_id,
        ] += (
            GYM_ALPHA
            *
            td_error
        )

        absolute_td_error += abs(
            td_error
        )

    print(
        "Epoch",
        epoch + 1,
        "Mean absolute TD error:",
        round(
            absolute_td_error
            /
            len(offline_states),
            6,
        ),
    )

Epoch 1 Mean absolute TD error: 0.473157
Epoch 2 Mean absolute TD error: 0.102486
Epoch 3 Mean absolute TD error: 0.063091
Epoch 4 Mean absolute TD error: 0.062142
Epoch 5 Mean absolute TD error: 0.062046
Epoch 6 Mean absolute TD error: 0.06203
Epoch 7 Mean absolute TD error: 0.062027
Epoch 8 Mean absolute TD error: 0.062026
Epoch 9 Mean absolute TD error: 0.062026
Epoch 10 Mean absolute TD error: 0.062026
Epoch 11 Mean absolute TD error: 0.062026
Epoch 12 Mean absolute TD error: 0.062026
Epoch 13 Mean absolute TD error: 0.062026
Epoch 14 Mean absolute TD error: 0.062026
Epoch 15 Mean absolute TD error: 0.062026


## 36. Extract the Gymnasium-Version Offline Policy

In [50]:
gym_policy = {}

for state_id in gym_observed_actions:

    supported = gym_observed_actions[
        state_id
    ]

    best_action = max(
        supported,
        key=lambda action_id:
            gym_q[
                state_id,
                action_id,
            ],
    )

    gym_policy[state_id] = best_action

## 37. Evaluate the Gymnasium Offline Policy

In [51]:
observation, info = env.reset(seed=0)

gym_path = [
    info["position"]
]

gym_return = 0
gym_success = False

for step_number in range(100):

    if observation not in gym_policy:
        break

    action = gym_policy[
        observation
    ]

    (
        observation,
        reward,
        terminated,
        truncated,
        info,
    ) = env.step(action)

    gym_return += reward

    gym_path.append(
        info["position"]
    )

    if terminated or truncated:

        gym_success = terminated
        break

print("Reached goal:", gym_success)
print("Return:", gym_return)
print("Steps:", len(gym_path) - 1)

print("\nPath:")
for position in gym_path:
    print(position)

Reached goal: True
Return: 10
Steps: 18

Path:
(0, 0)
(1, 0)
(2, 0)
(2, 1)
(3, 1)
(4, 1)
(5, 1)
(6, 1)
(6, 2)
(7, 2)
(8, 2)
(8, 3)
(8, 4)
(8, 5)
(8, 6)
(8, 7)
(8, 8)
(9, 8)
(9, 9)


## 38. Random Policy Baseline

In [52]:
RANDOM_EPISODES = 100

random_successes = 0
random_returns = []

for episode_number in range(RANDOM_EPISODES):

    observation, info = env.reset(
        seed=episode_number
    )

    episode_return = 0

    for step_number in range(100):

        action = env.action_space.sample()

        (
            observation,
            reward,
            terminated,
            truncated,
            info,
        ) = env.step(action)

        episode_return += reward

        if terminated:
            random_successes += 1
            break

    random_returns.append(
        episode_return
    )

print(
    "Random success rate:",
    random_successes
    /
    RANDOM_EPISODES
)

print(
    "Random average return:",
    sum(random_returns)
    /
    len(random_returns)
)

Random success rate: 0.03
Random average return: -136.55


## 39. Compare the Two Offline Implementations

Both versions use:

```text
same fixed CSV
same state IDs
same actions
same rewards
same support constraint
same Bellman update
same training hyperparameters
```

The difference is mainly software structure.

In [53]:
print("FROM SCRATCH")
print("Reached goal:", reached_goal)
print("Return:", total_reward)
print("Steps:", len(path) - 1)

print("\nGYMNASIUM VERSION")
print("Reached goal:", gym_success)
print("Return:", gym_return)
print("Steps:", len(gym_path) - 1)

FROM SCRATCH
Reached goal: True
Return: 10
Steps: 18

GYMNASIUM VERSION
Reached goal: True
Return: 10
Steps: 18


# Part IV — Interpretation

## 40. How Does the Reward Reach Earlier States?

Most valid transitions have immediate reward `0`.

The positive `+10` reward appears only when the goal is entered.

Q-learning repeatedly applies Bellman backups, so goal value propagates backward through earlier states.

That is how the agent can learn useful decisions at `(0,0)` even though its immediate reward there is zero.

## 41. Why Is This Offline RL?

Training uses:

```text
fixed CSV transitions
```

It does not do:

```text
choose training action
→ env.step()
→ collect a new transition
→ learn from it
```

Gymnasium is used only after the offline data already exist.

## 42. Limitation of the Current Reward

A normal movement costs `0`.

Therefore the learner is encouraged to:

```text
reach the goal
avoid collisions
```

but not explicitly to minimize the number of safe steps.

A future shortest-path version could assign:

```text
normal move = -0.1
```

## 43. Future Extensions

- randomize start and goal
- compare random vs expert offline datasets
- add Behavior Cloning
- add Conservative Q-Learning
- increase grid size
- use neural Q-functions
- add a small step penalty
- compare offline Q-learning with online Q-learning

# 44. Final Portfolio Takeaway

This project shows the complete offline RL pipeline:

```text
MDP definition
→ behavior data generation
→ fixed transition file
→ dataset inspection
→ offline Q updates
→ learned policy
→ Gymnasium evaluation
```

The core concept is that training experience is fixed before learning begins.